In [ ]:
import os
import random
import math
import time
import csv
from typing import Optional
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import sentencepiece as spm

random.seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

config_data_path = "./dataset/wmt_zh_en_training_corpus.csv"
config_checkpoint_dir = "./checkpoints"
os.makedirs(config_checkpoint_dir, exist_ok=True)

config_train_pairs: int = 4_350_000
config_val_pairs: int = 150_000

PAD_ID, UNK_ID, BOS_ID, EOS_ID = 0, 1, 2, 3

bpe_text_path = os.path.join(config_checkpoint_dir, "bpe_train.txt")
config_shared_vocab_size = 37_000

max_seq_len = 200
config_batch_size = 48
config_label_smoothing = 0.1
config_d_model = 512
config_n_heads = 8
config_n_encoder_layers = 6
config_n_decoder_layers = 6
config_d_ff = 2048
config_dropout = 0.1
config_warmup_steps = 4000
config_max_epochs = 12
config_grad_accum_steps = 10
config_log_every_steps = 50
config_grad_clip_norm = 1.0

config_adam_beta1: float = 0.9
config_adam_beta2: float = 0.98
config_adam_eps: float = 1e-9

config_qk_norm = True

# 混合精度用 bf16。bf16 的指数位和 fp32 一样多（8 bit），动态范围 3.4e38，
# 不存在 fp16 那个 65504 的上溢悬崖，也就不需要 GradScaler。
# 想做对照实验可以改成 torch.float16，代码会自动切回 GradScaler 路径。
config_amp_dtype = torch.bfloat16
config_use_amp = device.type == "cuda"
if config_amp_dtype == torch.bfloat16 and config_use_amp:
    assert torch.cuda.is_bf16_supported(), "bfloat16 is not supported on this GPU"

scaler = torch.amp.GradScaler('cuda', enabled=(config_use_amp and config_amp_dtype == torch.float16))
print(f"AMP: enabled={config_use_amp} dtype={config_amp_dtype} grad_scaler={scaler.is_enabled()}")

global_step = 0
best_val_loss = float("inf")


Device: cuda
GPU: NVIDIA GeForce RTX 5090
AMP: enabled=True dtype=torch.bfloat16 grad_scaler=False


In [ ]:
def load_and_sample_data(data_path: str, num_samples: int) -> list[tuple[str, str]]:
    reservoir: list[tuple[str, str]] = []
    total_lines = 0
    print(f"Sampling {num_samples:,} lines from {data_path}...")
    with open(data_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            if len(row) < 2: continue
            cn_text = row[0].strip()
            en_text = row[1].strip()
            if not cn_text or not en_text: continue
            total_lines += 1
            pair = (en_text, cn_text)
            if len(reservoir) < num_samples:
                reservoir.append(pair)
            else:
                j = random.randint(0, total_lines - 1)
                if j < num_samples:
                    reservoir[j] = pair
    print(f" Sampled {len(reservoir):,} pairs from {total_lines:,} total lines")
    return reservoir

print("Loading data...")
all_pairs = load_and_sample_data(config_data_path, (config_train_pairs + config_val_pairs))

class BPETokenizer:
    def __init__(self, model_path: str):
        self.sp = spm.SentencePieceProcessor()
        self.sp.load(model_path)

    @classmethod
    def train(cls, text_file: str, model_prefix: str, vocab_size: int = 37000) -> "BPETokenizer":
        spm.SentencePieceTrainer.train(
            input=text_file, model_prefix=model_prefix, vocab_size=vocab_size,
            model_type="bpe", character_coverage=0.9995,
            pad_id=PAD_ID, unk_id=UNK_ID, bos_id=BOS_ID, eos_id=EOS_ID,
            pad_piece="<pad>", unk_piece="<unk>", bos_piece="<bos>", eos_piece="<eos>",
            split_digits=True, byte_fallback=False, minloglevel=2,
        )
        return cls(f"{model_prefix}.model")

    def encode(self, text: str, add_bos: bool = True, add_eos: bool = True) -> list[int]:
        ids = self.sp.encode(text, out_type=int)
        if add_bos: ids = [BOS_ID] + ids
        if add_eos: ids = ids + [EOS_ID]
        return ids

    def decode(self, ids: list[int], join_char: str = "") -> str:
        filtered = [int(i) for i in ids if int(i) not in (PAD_ID, BOS_ID, EOS_ID)]
        text = self.sp.decode(filtered)
        if join_char: text = text.replace(" ", join_char)
        return text

    def vocab_size(self) -> int: return self.sp.vocab_size()

# BPE 分词
bpe_text_path = os.path.join(config_checkpoint_dir, "bpe_train.txt")
with open(bpe_text_path, "w", encoding="utf-8") as f:
    for en, cn in all_pairs:
        f.write(en + "\n")
        f.write("".join(cn.strip().split()) + "\n")

tokenizer = BPETokenizer.train(bpe_text_path, model_prefix=os.path.join(config_checkpoint_dir, "bpe"), vocab_size=config_shared_vocab_size)
config_shared_vocab_size = tokenizer.vocab_size()
print(f"Vocab size: {config_shared_vocab_size:,}")

# 数据预处理
random.shuffle(all_pairs)
full_train = all_pairs[:config_train_pairs]
full_val = all_pairs[config_train_pairs : config_train_pairs+config_val_pairs]

def process_texts(texts, tokenizer, max_len):
    results = []
    chunk_size = 500_000
    for i in range(0, len(texts), chunk_size):
        chunk = texts[i:i+chunk_size]
        en_texts = [en for en, _ in chunk]
        cn_texts = ["".join(cn.strip().split()) for _, cn in chunk]
        src_ids_batch = tokenizer.sp.encode(en_texts, out_type=int)
        tgt_ids_batch = tokenizer.sp.encode(cn_texts, out_type=int)

        for src_ids, tgt_ids in zip(src_ids_batch, tgt_ids_batch):
            if len(src_ids) > max_len - 2: src_ids = src_ids[:max_len-2]
            if len(tgt_ids) > max_len - 2: tgt_ids = tgt_ids[:max_len-2]
            src_final = [BOS_ID] + src_ids + [EOS_ID]
            tgt_final = [BOS_ID] + tgt_ids + [EOS_ID]
            results.append((torch.tensor(src_final, dtype=torch.long), torch.tensor(tgt_final, dtype=torch.long)))
    return results

tokenized_train = process_texts(full_train, tokenizer, max_seq_len)
tokenized_val = process_texts(full_val, tokenizer, max_seq_len)

# 封装 DataLoader
class PreTokenizedDataset(Dataset):
    def __init__(self, pairs): self.pairs = pairs
    def __len__(self): return len(self.pairs)
    def __getitem__(self, idx): return self.pairs[idx]

def collate_fn(batch: list[tuple[torch.Tensor, torch.Tensor]]) -> tuple[torch.Tensor, ...]:
    src_batch, tgt_batch = zip(*batch)
    src_lens = [len(s) for s in src_batch]
    tgt_lens = [len(t) for t in tgt_batch]
    src_max_len, tgt_max_len = max(src_lens), max(tgt_lens)

    src_padded = torch.full((len(src_batch), src_max_len), PAD_ID, dtype=torch.long)
    tgt_padded = torch.full((len(tgt_batch), tgt_max_len), PAD_ID, dtype=torch.long)

    for i, s in enumerate(src_batch): src_padded[i, :len(s)] = s
    for i, t in enumerate(tgt_batch): tgt_padded[i, :len(t)] = t

    tgt_input = tgt_padded[:, :-1]
    tgt_output = tgt_padded[:, 1:]

    src_key_padding_mask = (src_padded == PAD_ID)
    tgt_key_padding_mask = (tgt_input == PAD_ID)

    return src_padded, tgt_input, tgt_output, src_key_padding_mask, tgt_key_padding_mask

train_loader = DataLoader(PreTokenizedDataset(tokenized_train), 
        batch_size=config_batch_size, 
        shuffle=True, 
        collate_fn=collate_fn, 
        num_workers=4)
val_loader = DataLoader(PreTokenizedDataset(tokenized_val), 
        batch_size=config_batch_size, 
        shuffle=False, 
        collate_fn=collate_fn, 
        num_workers=4)

Loading data...
Sampling 4,500,000 lines from ./dataset/wmt_zh_en_training_corpus.csv...
 Sampled 4,500,000 pairs from 24,752,356 total lines
Vocab size: 37,000


In [3]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 5000, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0) # (1, max_len, d_model)
        self.register_buffer("pe", pe)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)

In [4]:
class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.linear2(self.dropout(torch.nn.functional.relu(self.linear1(x))))

In [ ]:
class MultiHeadAttention(nn.Module):
    # 基于 torch.nn.functional.scaled_dot_product_attention 的多头注意力，带可选的 QK-Norm。
    #
    # 掩码约定与 nn.MultiheadAttention 保持一致：传入的 attn_mask / key_padding_mask
    # 都是 True 表示遮蔽。注意 SDPA 的布尔 attn_mask 恰好相反（True 表示参与注意力），
    # 所以合并后要取反再交给 SDPA。

    # 类级开关：打开后 forward 会额外统计 max|score|，仅在要打印日志的那一步开启
    record_scores: bool = False

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1, qk_norm: bool = True):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        self.dropout_p = dropout

        # 三个投影分开写，而不是融合成一个 1536x512 的 in_proj_weight。
        # 这样 xavier_uniform_ 的 bound 是 sqrt(6/1024)，与手写版一致
        self.w_q = nn.Linear(d_model, d_model)
        self.w_k = nn.Linear(d_model, d_model)
        self.w_v = nn.Linear(d_model, d_model)
        self.w_o = nn.Linear(d_model, d_model)

        # QK-Norm：把 q/k 每个头的向量归一化，模长被钉在 sqrt(d_k) 量级，
        # 注意力 logit 在数学上就无法跑飞，不需要任何截断阈值
        self.q_norm = nn.LayerNorm(self.d_k) if qk_norm else nn.Identity()
        self.k_norm = nn.LayerNorm(self.d_k) if qk_norm else nn.Identity()

        # 最近一次 record_scores 打开时记录的 max|score|
        self.last_max_score = 0.0

    def _split_heads(self, x: torch.Tensor, batch_size: int) -> torch.Tensor:
        return x.view(batch_size, -1, self.n_heads, self.d_k).transpose(1, 2)

    def forward(
        self,
        query: torch.Tensor,
        key: torch.Tensor,
        value: torch.Tensor,
        attn_mask: Optional[torch.Tensor] = None,
        key_padding_mask: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        batch_size = query.size(0)

        q = self._split_heads(self.w_q(query), batch_size)
        k = self._split_heads(self.w_k(key), batch_size)
        v = self._split_heads(self.w_v(value), batch_size)

        q = self.q_norm(q)
        k = self.k_norm(k)

        # 合并因果掩码与 padding 掩码，True = 遮蔽
        masked: Optional[torch.Tensor] = None
        if attn_mask is not None:
            masked = attn_mask.view(1, 1, attn_mask.size(-2), attn_mask.size(-1))
        if key_padding_mask is not None:
            kpm = key_padding_mask.view(batch_size, 1, 1, -1)
            masked = kpm if masked is None else (masked | kpm)

        # 监控用：在 fp32 下显式算一次注意力分数，只统计不参与前向
        if MultiHeadAttention.record_scores:
            with torch.autocast(device_type=q.device.type, enabled=False):
                scores = torch.matmul(q.float(), k.float().transpose(-2, -1)) / math.sqrt(self.d_k)
                self.last_max_score = scores.abs().max().item()
                del scores

        # SDPA 的布尔掩码是 True = 参与注意力，与本模块的约定相反，故取反
        sdpa_mask = None if masked is None else ~masked
        out = torch.nn.functional.scaled_dot_product_attention(
            q, k, v,
            attn_mask=sdpa_mask,
            dropout_p=self.dropout_p if self.training else 0.0,
        )

        out = out.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.w_o(out)


class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1, qk_norm: bool = True):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout=dropout, qk_norm=qk_norm)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, src_key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        attn_out = self.self_attn(x, x, x, key_padding_mask=src_key_padding_mask)
        x = self.norm1(x + self.dropout(attn_out))
        ff_out = self.ff(x)
        x = self.norm2(x + self.dropout(ff_out))
        return x


class DecoderLayer(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, dropout: float = 0.1, qk_norm: bool = True):
        super().__init__()
        # 自注意力
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout=dropout, qk_norm=qk_norm)
        # 交叉注意力
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout=dropout, qk_norm=qk_norm)

        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, memory: torch.Tensor,
                tgt_mask: Optional[torch.Tensor] = None,
                tgt_key_padding_mask: Optional[torch.Tensor] = None,
                memory_key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:

        # 自注意力 (带因果掩码，True 表示遮蔽)
        attn_out = self.self_attn(x, x, x, attn_mask=tgt_mask, key_padding_mask=tgt_key_padding_mask)
        x = self.norm1(x + self.dropout(attn_out))

        # 交叉注意力 (Query 来自解码器，Key/Value 来自编码器)
        attn_out = self.cross_attn(x, memory, memory, key_padding_mask=memory_key_padding_mask)
        x = self.norm2(x + self.dropout(attn_out))

        # 前馈网络
        ff_out = self.ff(x)
        x = self.norm3(x + self.dropout(ff_out))
        return x


class Encoder(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, n_layers: int, dropout: float, qk_norm: bool = True):
        super().__init__()
        self.layers = nn.ModuleList([
            EncoderLayer(d_model, n_heads, d_ff, dropout, qk_norm) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor, src_key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, src_key_padding_mask=src_key_padding_mask)
        return self.norm(x)


class Decoder(nn.Module):
    def __init__(self, d_model: int, n_heads: int, d_ff: int, n_layers: int, dropout: float, qk_norm: bool = True):
        super().__init__()
        self.layers = nn.ModuleList([
            DecoderLayer(d_model, n_heads, d_ff, dropout, qk_norm) for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x: torch.Tensor, memory: torch.Tensor,
                tgt_mask: Optional[torch.Tensor] = None,
                tgt_key_padding_mask: Optional[torch.Tensor] = None,
                memory_key_padding_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x, memory, tgt_mask=tgt_mask,
                      tgt_key_padding_mask=tgt_key_padding_mask,
                      memory_key_padding_mask=memory_key_padding_mask)
        return self.norm(x)


In [6]:
class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size, d_model, n_heads, d_ff, n_encoder_layers, n_decoder_layers, dropout, qk_norm=True):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, dropout=dropout)

        self.encoder = Encoder(d_model, n_heads, d_ff, n_encoder_layers, dropout, qk_norm)
        self.decoder = Decoder(d_model, n_heads, d_ff, n_decoder_layers, dropout, qk_norm)

        self.fc_out = nn.Linear(d_model, vocab_size)
        # 权重共享
        self.fc_out.weight = self.embedding.weight

        # 记住这里一定要初始化
        # 注意 LayerNorm 的 weight/bias 和所有 bias 都是 1 维，会被跳过，保持 1 和 0
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def forward(self, src, tgt, src_key_padding_mask=None, tgt_key_padding_mask=None, tgt_mask=None):
        src_emb = self.embedding(src) * math.sqrt(self.d_model)
        tgt_emb = self.embedding(tgt) * math.sqrt(self.d_model)

        src_emb = self.pos_encoding(src_emb)
        tgt_emb = self.pos_encoding(tgt_emb)

        memory = self.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)
        outs = self.decoder(tgt_emb, memory, tgt_mask=tgt_mask, tgt_key_padding_mask=tgt_key_padding_mask, memory_key_padding_mask=src_key_padding_mask)

        return self.fc_out(outs)

model = Seq2SeqTransformer(
            config_shared_vocab_size,
            config_d_model,
            config_n_heads,
            config_d_ff,
            config_n_encoder_layers,
            config_n_decoder_layers,
            config_dropout,
            qk_norm=config_qk_norm).to(device)

print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")


Parameters: 63,126,152


In [7]:
# 自检：验证 MultiHeadAttention 的掩码取反是否正确。
# 关掉 QK-Norm 和 dropout 后，本模块必须与 nn.MultiheadAttention 数值一致。
def selfcheck_attention_matches_pytorch(d_model=64, n_heads=4, batch=3, seq=7, tol=1e-5):
    torch.manual_seed(0)
    mine = MultiHeadAttention(d_model, n_heads, dropout=0.0, qk_norm=False).to(device).eval()
    ref = nn.MultiheadAttention(d_model, n_heads, dropout=0.0, batch_first=True).to(device).eval()

    with torch.no_grad():
        ref.in_proj_weight.copy_(torch.cat([mine.w_q.weight, mine.w_k.weight, mine.w_v.weight], dim=0))
        ref.in_proj_bias.copy_(torch.cat([mine.w_q.bias, mine.w_k.bias, mine.w_v.bias], dim=0))
        ref.out_proj.weight.copy_(mine.w_o.weight)
        ref.out_proj.bias.copy_(mine.w_o.bias)

        x = torch.randn(batch, seq, d_model, device=device)
        key_padding_mask = torch.zeros(batch, seq, dtype=torch.bool, device=device)
        key_padding_mask[:, seq - 2:] = True                                  # True = 遮蔽
        causal = torch.triu(torch.ones(seq, seq, dtype=torch.bool, device=device), diagonal=1)

        for label, am, kpm in [("padding only", None, key_padding_mask),
                               ("causal only", causal, None),
                               ("causal + padding", causal, key_padding_mask)]:
            got = mine(x, x, x, attn_mask=am, key_padding_mask=kpm)
            want, _ = ref(x, x, x, attn_mask=am, key_padding_mask=kpm, need_weights=False)
            diff = (got - want).abs().max().item()
            status = "OK" if diff < tol else "FAIL"
            print(f"  [{status}] {label:18s} max abs diff vs nn.MultiheadAttention: {diff:.3e}")
            assert diff < tol, f"attention mismatch on '{label}': {diff}"

print("Self-check: attention vs nn.MultiheadAttention")
selfcheck_attention_matches_pytorch()
print("Self-check passed.")


Self-check: attention vs nn.MultiheadAttention
  [OK] padding only       max abs diff vs nn.MultiheadAttention: 1.490e-07
  [OK] causal only        max abs diff vs nn.MultiheadAttention: 1.490e-07
  [OK] causal + padding   max abs diff vs nn.MultiheadAttention: 1.490e-07
Self-check passed.


In [8]:
criterion = nn.CrossEntropyLoss(ignore_index=PAD_ID, label_smoothing=config_label_smoothing)

In [ ]:
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1.0,
    betas=(config_adam_beta1, config_adam_beta2),
    eps=config_adam_eps,
)


In [10]:
def lr_lambda(step: int):
    # torch.optim.lr_scheduler.LambdaLR 会从 step = 0 开始调用
    # 使用 step + 1 避免除零错误
    arg1 = (step + 1) ** (-0.5)
    arg2 = (step + 1) * (config_warmup_steps ** (-1.5))
    return config_d_model ** (-0.5) * min(arg1, arg2)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=[lr_lambda])

In [ ]:
def short_attn_name(name: str) -> str:
    """encoder.layers.0.self_attn -> enc0.self ; decoder.layers.3.cross_attn -> dec3.cross"""
    parts = name.split(".")
    side = "enc" if parts[0] == "encoder" else "dec"
    layer = parts[2] if len(parts) > 2 else "?"
    kind = parts[-1].replace("_attn", "")
    return f"{side}{layer}.{kind}"


def collect_attn_modules(model):
    return [(short_attn_name(n), m) for n, m in model.named_modules()
            if isinstance(m, MultiHeadAttention)]


def peak_attention_score(attn_modules):
    # 返回 (最大 |score|, 所在模块简称)。调用前需要 record_scores 已经打开过一次前向。
    best_val, best_name = 0.0, "-"
    for name, mod in attn_modules:
        if mod.last_max_score > best_val:
            best_val, best_name = mod.last_max_score, name
    return best_val, best_name


@torch.no_grad()
def validate(model, dataloader, criterion, device):
    model.eval()
    total_loss, total_tokens = 0.0, 0
    for batch in dataloader:
        src, tgt_input, tgt_output, src_mask, tgt_mask = [x.to(device, non_blocking=True) for x in batch]
        tgt_seq_len = tgt_input.size(1)
        causal_mask = torch.triu(torch.ones(tgt_seq_len, tgt_seq_len, dtype=torch.bool, device=device), diagonal=1)

        with torch.amp.autocast('cuda', dtype=config_amp_dtype, enabled=config_use_amp):
            logits = model(src, tgt_input, src_key_padding_mask=src_mask, tgt_key_padding_mask=tgt_mask, tgt_mask=causal_mask)
            loss = criterion(logits.reshape(-1, logits.size(-1)).float(), tgt_output.reshape(-1))

        n_tokens = (tgt_output != PAD_ID).sum().item()
        total_loss += loss.item() * n_tokens
        total_tokens += n_tokens

    model.train()
    return total_loss / max(total_tokens, 1)


def train_epoch(model, dataloader, optimizer, criterion, device, global_step, epoch, scaler=None):
    model.train()
    use_scaler = scaler is not None and scaler.is_enabled()
    attn_modules = collect_attn_modules(model)

    # 整个 epoch 的累计（只用于 epoch summary）
    total_loss = torch.tensor(0.0, device=device)
    total_tokens = torch.tensor(0, device=device)
    # 打印窗口的累计，每次打印后重置。
    # 原版用整个 epoch 的累计平均，一个 micro-batch 出 NaN 会锁死本 epoch 剩下所有日志行。
    step_loss = torch.tensor(0.0, device=device)
    step_tokens = torch.tensor(0, device=device)

    accum_count = 0
    nonfinite_batches = 0
    skipped_updates = 0
    epoch_start = time.time()
    lr = optimizer.param_groups[0]["lr"]
    attn_peak, attn_where = 0.0, "-"

    for batch_idx, batch in enumerate(dataloader):
        src, tgt_input, tgt_output, src_mask, tgt_mask = [x.to(device, non_blocking=True) for x in batch]

        # 生成因果掩码 ，True 表示遮蔽
        tgt_seq_len = tgt_input.size(1)
        causal_mask = torch.triu(torch.ones(tgt_seq_len, tgt_seq_len, dtype=torch.bool, device=device), diagonal=1)

        is_log_batch = (batch_idx + 1) % config_log_every_steps == 0
        MultiHeadAttention.record_scores = is_log_batch

        with torch.amp.autocast('cuda', dtype=config_amp_dtype, enabled=config_use_amp):
            logits = model(src, tgt_input,
                           src_key_padding_mask=src_mask,
                           tgt_key_padding_mask=tgt_mask,
                           tgt_mask=causal_mask)
            loss = criterion(logits.reshape(-1, logits.size(-1)).float(), tgt_output.reshape(-1))
            loss = loss / config_grad_accum_steps

        MultiHeadAttention.record_scores = False
        if is_log_batch:
            attn_peak, attn_where = peak_attention_score(attn_modules)

        # 非有限值守卫：出现 NaN/Inf 就直接跳过这个 micro-batch，
        # 不做 backward，已经累积的梯度不受污染
        if not torch.isfinite(loss):
            nonfinite_batches += 1
            if nonfinite_batches <= 5:
                print(f" [warn] non-finite loss at epoch {epoch} batch {batch_idx + 1}, micro-batch skipped")
            del logits, loss
        else:
            if use_scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            n_tokens = (tgt_output != PAD_ID).sum()
            contrib = loss.detach() * config_grad_accum_steps * n_tokens
            total_loss += contrib
            total_tokens += n_tokens
            step_loss += contrib
            step_tokens += n_tokens
            accum_count += 1

        # 参数更新
        should_step = ((batch_idx + 1) % config_grad_accum_steps == 0) or ((batch_idx + 1) == len(dataloader))
        if should_step and accum_count > 0:
            if use_scaler:
                scaler.unscale_(optimizer)

            # 累积组不完整时把梯度放大回一个完整组的量级
            if accum_count != config_grad_accum_steps:
                correction = config_grad_accum_steps / accum_count
                for p in model.parameters():
                    if p.grad is not None:
                        p.grad.mul_(correction)

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=config_grad_clip_norm)

            if use_scaler:
                before = scaler.get_scale()
                scaler.step(optimizer)
                scaler.update()
                if scaler.get_scale() < before:
                    skipped_updates += 1
            else:
                optimizer.step()

            optimizer.zero_grad(set_to_none=True)
            scheduler.step()
            global_step += 1
            accum_count = 0
            lr = scheduler.get_last_lr()[0]

        # 日志打印
        if is_log_batch:
            elapsed = time.time() - epoch_start
            steps_done = batch_idx + 1
            eta_seconds = (elapsed / steps_done) * (len(dataloader) - steps_done) if steps_done > 0 else 0.0

            window_tokens = max(step_tokens.item(), 1)
            avg_loss = step_loss.item() / window_tokens
            ppl = math.exp(avg_loss) if math.isfinite(avg_loss) and avg_loss < 20 else float("nan")
            ppl_str = f"{ppl:.1f}" if math.isfinite(ppl) else "n/a"

            print(f" Epoch {epoch:2d}/{config_max_epochs} | Step {global_step:6d} | "
                  f"Batch {steps_done:5d}/{len(dataloader)} | "
                  f"Loss {avg_loss:.4f} | PPL {ppl_str} | LR {lr:.2e} | "
                  f"AttnMax {attn_peak:.1f} @{attn_where} | "
                  f"ETA {int(eta_seconds // 60)}m{int(eta_seconds % 60)}s")

            step_loss = torch.tensor(0.0, device=device)
            step_tokens = torch.tensor(0, device=device)

    avg_loss = total_loss.item() / max(total_tokens.item(), 1)
    ppl = math.exp(avg_loss) if math.isfinite(avg_loss) and avg_loss < 20 else float("nan")
    return global_step, avg_loss, ppl, nonfinite_batches, skipped_updates


In [12]:
def state_dict_is_finite(state_dict) -> bool:
    return all(torch.isfinite(v).all().item() for v in state_dict.values() if v.is_floating_point())


def save_ckpt(path, model, optimizer, scheduler, scaler, step, epoch, best_val_loss):
    # 保存前检查参数是否有限。之前 checkpoints/latest.pt 就是一个 186 个张量全 NaN 的存档：
    # val_loss 是 NaN 时 `val_loss < best_val_loss` 为 False，于是走 else 分支照存不误。
    sd = model.state_dict()
    if not state_dict_is_finite(sd):
        raise RuntimeError(f"refusing to save non-finite model state to {path}")
    torch.save({
        'model': sd,
        'optimizer': optimizer.state_dict(),
        'scheduler': scheduler.state_dict(),
        'scaler': scaler.state_dict() if scaler is not None else None,
        'step': step,
        'epoch': epoch,
        'best_val_loss': best_val_loss,
    }, path)


print("\n--- Start Training ---")
import gc

for epoch in range(1, config_max_epochs + 1):
    print(f"\n--- Epoch {epoch}/{config_max_epochs} ---")
    gc.collect()

    global_step, train_loss, train_ppl, nonfinite_batches, skipped_updates = train_epoch(
        model, train_loader, optimizer, criterion, device, global_step, epoch, scaler)

    val_loss = validate(model, val_loader, criterion, device)
    val_ppl = math.exp(val_loss) if math.isfinite(val_loss) and val_loss < 20 else float("nan")

    attn_peak, attn_where = 0.0, "-"
    print(f"Epoch {epoch} Summary | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
          f"Val PPL: {val_ppl:.1f} | non-finite micro-batches: {nonfinite_batches} | "
          f"skipped updates: {skipped_updates}")

    if not math.isfinite(val_loss):
        print(" Validation loss is not finite, stopping.")
        break

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_ckpt(os.path.join(config_checkpoint_dir, "best_model.pt"),
                  model, optimizer, scheduler, scaler, global_step, epoch, best_val_loss)
        print(" New best model saved.")
    else:
        save_ckpt(os.path.join(config_checkpoint_dir, "latest.pt"),
                  model, optimizer, scheduler, scaler, global_step, epoch, best_val_loss)



--- Start Training ---

--- Epoch 1/12 ---
 Epoch  1/12 | Step      5 | Batch    50/90625 | Loss 10.5265 | PPL 37289.0 | LR 1.05e-06 | AttnMax 5.1 @enc3.self | ETA 140m38s
 Epoch  1/12 | Step     10 | Batch   100/90625 | Loss 10.5135 | PPL 36809.1 | LR 1.92e-06 | AttnMax 5.5 @dec1.cross | ETA 130m43s
 Epoch  1/12 | Step     15 | Batch   150/90625 | Loss 10.4876 | PPL 35867.3 | LR 2.80e-06 | AttnMax 5.3 @enc3.self | ETA 126m24s
 Epoch  1/12 | Step     20 | Batch   200/90625 | Loss 10.4547 | PPL 34706.8 | LR 3.67e-06 | AttnMax 5.1 @enc3.self | ETA 123m31s
 Epoch  1/12 | Step     25 | Batch   250/90625 | Loss 10.4144 | PPL 33334.9 | LR 4.54e-06 | AttnMax 4.7 @dec3.cross | ETA 121m39s
 Epoch  1/12 | Step     30 | Batch   300/90625 | Loss 10.3789 | PPL 32174.6 | LR 5.42e-06 | AttnMax 4.7 @enc1.self | ETA 118m37s
 Epoch  1/12 | Step     35 | Batch   350/90625 | Loss 10.3477 | PPL 31184.9 | LR 6.29e-06 | AttnMax 4.8 @enc2.self | ETA 116m43s
 Epoch  1/12 | Step     40 | Batch   400/90625 | Lo

In [13]:
# 数值余量检查：跑若干个真实 batch，打印每个注意力模块每个头的 max|score|。
# 判定标准：全部 < 500，且没有单个 head 比同层其它 head 高两个数量级。
@torch.no_grad()
def report_attention_scores(model, dataloader, n_batches=20):
    model.eval()
    stats = {}
    handles = []

    def make_hook(name, mod):
        def hook(_m, inputs, _out):
            query, key = inputs[0], inputs[1]
            with torch.autocast(device_type=query.device.type, enabled=False):
                b = query.size(0)
                q = mod._split_heads(mod.w_q(query.float()), b)
                k = mod._split_heads(mod.w_k(key.float()), b)
                q, k = mod.q_norm(q), mod.k_norm(k)
                sc = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(mod.d_k)
                per_head = sc.abs().amax(dim=(0, 2, 3)).cpu()
            prev = stats.get(name)
            stats[name] = per_head if prev is None else torch.maximum(prev, per_head)
        return hook

    for name, mod in model.named_modules():
        if isinstance(mod, MultiHeadAttention):
            handles.append(mod.register_forward_hook(make_hook(name, mod)))

    try:
        for i, batch in enumerate(dataloader):
            if i >= n_batches:
                break
            src, tgt_input, _, src_mask, tgt_mask = [x.to(device, non_blocking=True) for x in batch]
            n = tgt_input.size(1)
            causal_mask = torch.triu(torch.ones(n, n, dtype=torch.bool, device=device), diagonal=1)
            with torch.amp.autocast('cuda', dtype=config_amp_dtype, enabled=config_use_amp):
                model(src, tgt_input, src_key_padding_mask=src_mask, tgt_key_padding_mask=tgt_mask, tgt_mask=causal_mask)
    finally:
        for h in handles:
            h.remove()
        model.train()

    worst = 0.0
    print(f"{'module':34s} {'max|score|':>12s}   per-head")
    for name in sorted(stats):
        heads = stats[name]
        worst = max(worst, heads.max().item())
        print(f"{name:34s} {heads.max().item():12.2f}   " + " ".join(f"{v:7.2f}" for v in heads.tolist()))
    print(f"\nglobal max |score| = {worst:.2f}   (expected < 500 with QK-Norm; "
          f"fp16 overflow threshold in the hand-written version is 8188)")
    return worst

report_attention_scores(model, val_loader)


module                               max|score|   per-head
decoder.layers.0.cross_attn               19.18     16.19   16.39   19.18   16.56   16.54   16.14   18.88   18.66
decoder.layers.0.self_attn                21.99     16.12   11.86   21.99   13.41   11.05   10.54   15.76   17.87
decoder.layers.1.cross_attn               22.54     14.65   17.53   13.06   13.44   17.53   12.41   16.97   22.54
decoder.layers.1.self_attn                18.77     18.57   16.77   18.77   16.37   10.07   15.26   15.89   12.48
decoder.layers.2.cross_attn               25.74     15.41   15.45   17.38   18.75   25.74   19.32   22.25   22.05
decoder.layers.2.self_attn                15.86     12.64   10.18   13.58   15.86   12.21   12.13   14.70   12.49
decoder.layers.3.cross_attn               36.37     36.37   32.95   29.45   34.00   34.73   33.64   35.80   30.71
decoder.layers.3.self_attn                16.25      9.18    8.37   13.42    8.73   16.25   13.92   11.97   12.34
decoder.layers.4.cross_attn  

223.75674438476562

In [14]:
@torch.no_grad()
def translate(model, tokenizer, text, max_len=256, beam_size=4, length_penalty=0.6, device=torch.device("cuda")):
    model.eval()
    src_ids = tokenizer.encode(text)[:max_len]
    src_tensor = torch.tensor([src_ids], dtype=torch.long).to(device)
    src_key_padding_mask = torch.zeros_like(src_tensor, dtype=torch.bool).to(device)

    # 编码
    src_emb = model.embedding(src_tensor) * math.sqrt(model.d_model)
    src_emb = model.pos_encoding(src_emb)
    memory = model.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)

    # 束搜索
    beams = [([BOS_ID], 0.0, False)]
    completed = []

    for _ in range(max_len):
        if not beams: break
        candidates = []
        for tokens, score, finished in beams:
            if finished:
                candidates.append((tokens, score, True))
                continue
            if tokens[-1] == EOS_ID:
                completed.append((tokens, score))
                candidates.append((tokens, score, True))
                continue

            tgt_tensor = torch.tensor([tokens], dtype=torch.long).to(device)
            # 生成当前步的因果掩码
            causal_mask = torch.triu(torch.ones(len(tokens), len(tokens), dtype=torch.bool, device=device), diagonal=1)

            # 解码
            tgt_emb = model.embedding(tgt_tensor) * math.sqrt(model.d_model)
            tgt_emb = model.pos_encoding(tgt_emb)
            decoder_out = model.decoder(tgt_emb, memory, tgt_mask=causal_mask, memory_key_padding_mask=src_key_padding_mask)
            logits = model.fc_out(decoder_out[:, -1, :])

            log_probs = torch.log_softmax(logits, dim=-1)
            log_probs[0][UNK_ID] = float("-inf")

            topk_log_probs, topk_indices = torch.topk(log_probs[0], beam_size)
            for i in range(beam_size):
                new_score = score + topk_log_probs[i].item()
                new_tokens = tokens + [topk_indices[i].item()]
                candidates.append((new_tokens, new_score, topk_indices[i].item() == EOS_ID))

        candidates.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)
        beams = candidates[:beam_size]
        if all(f for _, _, f in beams): break

    if not completed:
        for t, s, _ in beams: completed.append((t, s))
    completed.sort(key=lambda x: x[1] / (len(x[0]) ** length_penalty), reverse=True)

    best_ids = [i for i in completed[0][0] if i not in (BOS_ID, EOS_ID, PAD_ID)]
    return tokenizer.decode(best_ids), completed[0][1]

In [15]:
model.load_state_dict(torch.load(os.path.join(config_checkpoint_dir, "best_model.pt"), map_location=device)['model'])
test_sentences = ["The weather is nice today .", "I have two brothers ."]
for sent in test_sentences:
    res, score = translate(model, tokenizer, sent, device=device)
    print(f"EN: {sent}\nCN: {res}\nScore: {score:.2f}\n")


EN: The weather is nice today .
CN: 今天天气很好。
Score: -1.91

EN: I have two brothers .
CN: 我有两个兄弟。
Score: -1.40



In [16]:
long_sent = "The fact that the scientist who the committee had appointed to oversee the project which was funded by the government ignored the safety protocols resulted in a catastrophic failure."
res, score = translate(model, tokenizer, long_sent, device=device)
print(f"EN: {long_sent}\nCN: {res}\nScore: {score:.2f}\n")

long_sent = "I have nothing in common with lazy people who blame others for their lack of success; I believe that if you are not willing to learn, no one can help you, and if you are determined to learn, no one can stop you."
res, score = translate(model, tokenizer, long_sent, device=device)
print(f"EN: {long_sent}\nCN: {res}\nScore: {score:.2f}\n")

EN: The fact that the scientist who the committee had appointed to oversee the project which was funded by the government ignored the safety protocols resulted in a catastrophic failure.
CN: 委员会任命监督由政府资助的项目的科学家忽视了安全协议,从而造成了灾难性的失败。
Score: -14.73

EN: I have nothing in common with lazy people who blame others for their lack of success; I believe that if you are not willing to learn, no one can help you, and if you are determined to learn, no one can stop you.
CN: 我和懒散的人没有什么共同之处,他们因为失败而责备别人,我相信如果你不愿意学习,没有人能帮助你,如果你决心学习,没有人能阻止你。
Score: -23.23

